# 🐘 Lab Semana 1 — Tipos de Datos y Arquitectura Big Data
## Big Data DD283 | Universidad Autónoma del Perú | 2026-1

---
**Nombre**: _(escribe tu nombre aquí)_  
**Grupo de proyecto**: _(nombre del grupo)_  
**Empresa/sector del proyecto**: _(empresa que analizarán)_  
**Fecha**: _(fecha de entrega)_

---

### 🎯 Objetivos del Lab
Al completar este laboratorio podrás:
1. Cargar y explorar datos **estructurados** con Pandas
2. Procesar datos **semi-estructurados** (JSON)
3. Analizar datos **no estructurados** (texto)
4. Calcular métricas de las **5 V's** del Big Data
5. Diseñar una **arquitectura Big Data** básica para tu proyecto

### ⏱️ Tiempo estimado: 90 minutos

### 📋 Instrucciones
- Ejecuta cada celda en orden (Shift+Enter)
- Donde veas `# 📝 TU CÓDIGO AQUÍ`, escribe tu implementación
- La última celda es una reflexión personal — escríbela con honestidad
- Sube el notebook completo (con outputs) a tu PR de GitHub

In [ ]:
# ── Celda 0: Instalación (solo si estás en Google Colab) ──────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install pandas matplotlib seaborn -q
    print('✅ Librerías instaladas en Colab')
else:
    print('✅ Usando entorno local (conda bigdata-ua)')

In [ ]:
# ── Celda 1: Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime, timedelta

# Estilo de gráficos
plt.style.use('dark_background')
sns.set_palette('husl')

print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'Python  : {sys.version.split()[0]}')
print('✅ Entorno listo para el lab S1!')

---
## PARTE 1: Datos Estructurados (25 min)
### Dataset: Transacciones de empresa peruana (100K registros)

In [ ]:
# ── Celda 2: Generar dataset estructurado ─────────────────────
np.random.seed(42)
N = 100_000  # 100 mil registros

departamentos = ['Lima', 'Arequipa', 'Cusco', 'Trujillo', 'Piura', 'Chiclayo', 'Iquitos', 'Huancayo']
categorias    = ['Alimentos', 'Electrodomésticos', 'Ropa', 'Farmacia', 'Tecnología', 'Deportes']
metodos_pago  = ['Yape', 'Plin', 'Tarjeta', 'Efectivo', 'BIM']

df = pd.DataFrame({
    'id':          range(1, N+1),
    'fecha':       pd.date_range('2024-01-01', periods=N, freq='5min'),
    'cliente_id':  np.random.randint(1000, 9999, N),
    'depto':       np.random.choice(departamentos, N, p=[.45,.12,.10,.09,.08,.07,.05,.04]),
    'categoria':   np.random.choice(categorias, N),
    'monto':       np.round(np.random.exponential(180, N) + 20, 2),
    'metodo_pago': np.random.choice(metodos_pago, N, p=[.30,.20,.25,.20,.05]),
    'es_fraude':   np.random.choice([0,1], N, p=[.98,.02]),
})

print(f'📊 Dataset creado: {df.shape[0]:,} registros × {df.shape[1]} columnas')
print(f'💾 Tamaño en memoria: {df.memory_usage(deep=True).sum()/1024:.1f} KB')
df.head(3)

In [ ]:
# ── Celda 3: Exploración básica ───────────────────────────────
print('=== INFORMACIÓN GENERAL ===')
df.info()
print()
print('=== ESTADÍSTICAS DESCRIPTIVAS ===')
df[['monto', 'es_fraude']].describe()

In [ ]:
# ── Celda 4: Análisis de las 5 V's ────────────────────────────
print('=== ANÁLISIS DE LAS 5 V\u2019S ===')

# VOLUMEN
mem_kb = df.memory_usage(deep=True).sum() / 1024
print(f'📦 VOLUMEN:')
print(f'   Registros:  {len(df):,}')
print(f'   En memoria: {mem_kb:.1f} KB')
print(f'   Escalando a 100M registros → {mem_kb*1000/1024**2:.1f} GB')

# VELOCIDAD
trans_dia = df.groupby(df['fecha'].dt.date).size()
print(f'\n⚡ VELOCIDAD:')
print(f'   Transacciones/día promedio: {trans_dia.mean():.0f}')
print(f'   Pico máximo en un día:      {trans_dia.max()}')

# VERACIDAD
print(f'\n✅ VERACIDAD:')
print(f'   Nulos por columna:')
print(df.isnull().sum().to_string())
print(f'   Tasa de fraude: {df["es_fraude"].mean()*100:.2f}%')

In [ ]:
# ── Celda 5: Visualización ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Dashboard Transacciones — Empresa Retail Peruana', fontsize=14, fontweight='bold', color='white')

# Gráfico 1: Ventas por departamento
ventas_depto = df.groupby('depto')['monto'].sum().sort_values(ascending=True)
axes[0,0].barh(ventas_depto.index, ventas_depto.values, color='#EB7B2F')
axes[0,0].set_title('Ventas por Departamento', color='white')
axes[0,0].set_xlabel('Monto Total (S/.)', color='white')

# Gráfico 2: Métodos de pago
mp = df['metodo_pago'].value_counts()
colors = ['#EB7B2F','#4BACC6','#70AD47','#FFD700','#C0504D']
axes[0,1].pie(mp.values, labels=mp.index, autopct='%1.1f%%', colors=colors)
axes[0,1].set_title('Métodos de Pago', color='white')

# Gráfico 3: Ventas por categoría
cat_ventas = df.groupby('categoria')['monto'].sum().sort_values()
axes[1,0].barh(cat_ventas.index, cat_ventas.values, color='#4BACC6')
axes[1,0].set_title('Ventas por Categoría', color='white')

# Gráfico 4: Tendencia mensual
monthly = df.groupby(df['fecha'].dt.month)['monto'].sum()
axes[1,1].plot(monthly.index, monthly.values, 'o-', color='#70AD47', linewidth=2, markersize=8)
axes[1,1].set_title('Tendencia Mensual de Ventas', color='white')
axes[1,1].set_xlabel('Mes', color='white')
axes[1,1].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig('dashboard_s1.png', dpi=150, bbox_inches='tight', facecolor='#0D1B3E')
plt.show()
print('✅ Dashboard guardado como dashboard_s1.png')

---
## PARTE 2: Datos Semi-Estructurados — JSON (15 min)

In [ ]:
# ── Celda 6: Trabajar con JSON ────────────────────────────────
# Simula la respuesta de una API de clientes (como la API de Reniec o SUNAT)

api_response = '''
[
  {"id": "CLI-001", "nombre": "Juan Quispe", "dni": "45678901",
   "contacto": {"email": "j.quispe@empresa.pe", "tel": "987654321", "distrito": "San Miguel"},
   "compras": [150.5, 320.0, 89.9], "activo": true, "tags": ["premium", "digital"]},
  {"id": "CLI-002", "nombre": "Maria Torres", "dni": "56789012",
   "contacto": {"email": null, "tel": "976543210", "distrito": "Miraflores"},
   "compras": [500.0, 1200.5], "activo": true, "tags": ["vip"]},
  {"id": "CLI-003", "nombre": "Carlos Mamani", "dni": "67890123",
   "contacto": {"email": "c.mamani@gmail.com", "tel": null, "distrito": "Callao"},
   "compras": [], "activo": false, "tags": ["nuevo", "inactivo"]}
]
'''

clientes = json.loads(api_response)
print(f'Clientes recibidos: {len(clientes)}')
print(f'\nEstructura del primer cliente:')
print(json.dumps(clientes[0], indent=2, ensure_ascii=False))

In [ ]:
# ── Celda 7: Normalizar JSON a DataFrame ──────────────────────
registros = []
for c in clientes:
    registros.append({
        'id': c['id'],
        'nombre': c['nombre'],
        'distrito': c['contacto']['distrito'],
        'tiene_email': c['contacto']['email'] is not None,
        'tiene_tel': c['contacto']['tel'] is not None,
        'num_compras': len(c['compras']),
        'total_compras': sum(c['compras']),
        'activo': c['activo'],
        'es_premium': any(t in ['premium','vip'] for t in c['tags'])
    })

df_clientes = pd.DataFrame(registros)
print('=== CLIENTES NORMALIZADOS ===')
print(df_clientes.to_string(index=False))
print(f'\n📊 Completitud de datos:')
print(f'  Con email: {df_clientes["tiene_email"].sum()}/{len(df_clientes)}')
print(f'  Con tel:   {df_clientes["tiene_tel"].sum()}/{len(df_clientes)}')
print(f'  Activos:   {df_clientes["activo"].sum()}/{len(df_clientes)}')

---
## PARTE 3: Datos No Estructurados — Análisis de Texto (20 min)

In [ ]:
# ── Celda 8: Análisis básico de sentimiento en comentarios ────
comentarios = [
    {'id': 1, 'texto': 'Excelente servicio en Plaza Vea, muy rápido el pago con Yape!', 'fuente': 'Google'},
    {'id': 2, 'texto': 'Pésima atención, esperé 45 minutos para ser atendido en Tottus', 'fuente': 'Facebook'},
    {'id': 3, 'texto': 'El producto llegó a tiempo pero la caja estaba dañada. Regular', 'fuente': 'App'},
    {'id': 4, 'texto': 'Super bueno el envío express, llegó en 2 horas a Miraflores', 'fuente': 'Google'},
    {'id': 5, 'texto': 'Me cobraron de más y tardaron 1 semana en devolver el dinero', 'fuente': 'Twitter'},
    {'id': 6, 'texto': 'Personal muy amable y precios competitivos, volvería sin duda', 'fuente': 'Google'},
    {'id': 7, 'texto': 'App del supermercado se cae constantemente, muy mala experiencia', 'fuente': 'AppStore'},
    {'id': 8, 'texto': 'Calidad excelente pero precio un poco alto comparado con Metro', 'fuente': 'Google'},
]

POS = {'excelente','super','bueno','rápido','amable','bien','tiempo','sin duda'}
NEG = {'pésima','mala','dañada','tardaron','mal','cae','alto','45 minutos'}

resultados = []
for c in comentarios:
    texto_lower = c['texto'].lower()
    pos = sum(1 for p in POS if p in texto_lower)
    neg = sum(1 for n in NEG if n in texto_lower)
    sentimiento = 'POSITIVO' if pos > neg else ('NEGATIVO' if neg > pos else 'NEUTRO')
    resultados.append({'id': c['id'], 'sentimiento': sentimiento,
                       'palabras': len(c['texto'].split()),
                       'fuente': c['fuente'],
                       'texto': c['texto'][:40]+'...'})

df_sent = pd.DataFrame(resultados)
print('=== ANÁLISIS DE SENTIMIENTO ===')
print(df_sent.to_string(index=False))
print(f'\n📊 Distribución:')
print(df_sent['sentimiento'].value_counts().to_string())

---
## PARTE 4: Arquitectura Big Data para Tu Proyecto (30 min)

### 📝 INSTRUCCIÓN:
Completa el diccionario `mi_arquitectura` con los datos de **TU PROYECTO**.
No uses el ejemplo — usa la empresa y el problema de tu grupo.

In [ ]:
# ── Celda 9: Diseñar arquitectura de TU proyecto ──────────────
# 📝 COMPLETA CON TU PROYECTO:

mi_arquitectura = {
    # ── INFORMACIÓN DEL PROYECTO ──────────────────────────────
    "proyecto": "[NOMBRE DE TU PROYECTO]",
    "empresa": "[EMPRESA O SECTOR]",
    "problema": "[DESCRIBE EL PROBLEMA EN 1-2 ORACIONES]",
    
    # ── LAS 5 V's DE TU PROYECTO ──────────────────────────────
    "5Vs": {
        "Volumen": "[Estimación: ¿cuántos registros/día? ¿TB, PB?]",
        "Velocidad": "[¿Batch, Near Real-time, Real-time?]",
        "Variedad": "[¿Qué tipos de datos? Estructurado/Semi/No]",
        "Veracidad": "[¿Qué problemas de calidad esperas?]",
        "Valor": "[¿Qué decisión de negocio habilita el análisis?]",
    },
    
    # ── ARQUITECTURA TÉCNICA ──────────────────────────────────
    "fuentes_datos": [
        "[Fuente 1: ej. Base de datos de ventas]",
        "[Fuente 2: ej. API de pagos Yape]",
        "[Fuente 3: ej. Redes sociales]",
    ],
    "ingesta": "[Herramienta: Kafka, Sqoop, API batch, etc.]",
    "almacenamiento": "[Herramienta: HDFS, MongoDB, S3, etc.]",
    "procesamiento": "[Herramienta: Spark, MapReduce, Hive, etc.]",
    "analisis": "[Herramienta: PySpark ML, Scikit-learn, etc.]",
    "visualizacion": "[Herramienta: Power BI, Tableau, Matplotlib, etc.]",
    
    # ── CASOS DE USO ─────────────────────────────────────────
    "casos_uso": [
        "[Caso 1: ej. Detección de fraude en tiempo real]",
        "[Caso 2: ej. Recomendaciones personalizadas]",
        "[Caso 3: ej. Predicción de demanda]",
    ],
    
    # ── CONSIDERACIONES ÉTICAS ───────────────────────────────
    "etica": "[¿Qué consideraciones de privacidad y ética hay?]",
}

print('=== ARQUITECTURA BIG DATA — MI PROYECTO ===')
print(json.dumps(mi_arquitectura, indent=2, ensure_ascii=False))

In [ ]:
# ── Celda 10: Calcular estimaciones de volumen ─────────────────
# 📝 ADAPTA ESTOS NÚMEROS A TU PROYECTO:

# Ejemplo: empresa de telecomunicaciones
usuarios         = 8_000_000   # Cambia por tu estimación
registros_x_dia  = 500         # Registros por usuario por día
tamaño_registro  = 1           # KB por registro

# Cálculos
registros_dia  = usuarios * registros_x_dia
registros_año  = registros_dia * 365
gb_dia         = (registros_dia * tamaño_registro) / (1024**2)
tb_año         = gb_dia * 365 / 1024

print('=== ESTIMACIÓN DE VOLUMEN ===')
print(f'Usuarios/clientes:        {usuarios:>15,}')
print(f'Registros/día:            {registros_dia:>15,}')
print(f'Registros/año:            {registros_año:>15,}')
print(f'Tamaño/día:               {gb_dia:>14.1f} GB')
print(f'Tamaño/año:               {tb_año:>14.1f} TB')
print()
if tb_año > 1:
    print('✅ ESTO ES BIG DATA — Un solo servidor no puede manejar esto.')
    print('   Necesitas un cluster distribuido (Hadoop/Spark).')
else:
    print('⚠️  Con estos volúmenes podrías usar una BD tradicional.')
    print('   Considera si tu proyecto tiene potencial de escalar.')

---
## ✍️ REFLEXIÓN FINAL (obligatorio — escribe con tus palabras)

Esta celda es evaluada por el docente. Responde honestamente, con tus propias palabras.
No copies de internet — el docente puede detectarlo.

### 🔍 Mis aprendizajes de esta semana:

**1. ¿Cuál fue el concepto más importante que aprendí?**
_(Escribe aquí — mínimo 3 oraciones)_

**2. ¿Cómo se conecta Big Data con mi trabajo actual?**
_(Describe un problema real de tu empresa que podría resolverse con Big Data)_

**3. ¿Qué fue lo más difícil del laboratorio?**
_(¿Qué parte te costó más y cómo la resolviste?)_

**4. ¿Por qué elegimos el proyecto que elegimos?**
_(¿Qué problema real resuelve? ¿Por qué es relevante para el mercado peruano?)_

**5. ¿Qué pregunta me quedó sin responder?**
_(Una pregunta técnica específica sobre Big Data que quieres explorar)_

In [ ]:
# ── Celda final: Verificación de entrega ──────────────────────
import os

print('=== CHECKLIST DE ENTREGA ===')
checks = [
    ('Nombre escrito en la primera celda',          True),  # Cambia a False si no lo hiciste
    ('Todas las celdas ejecutadas sin error',        True),
    ('Gráfico dashboard_s1.png generado',            os.path.exists('dashboard_s1.png')),
    ('JSON analizado correctamente (Celda 7)',        True),
    ('Análisis de sentimiento completado (Celda 8)', True),
    ('Arquitectura de mi proyecto completada',       True),  # Cambia a False si no lo completaste
    ('Reflexión final escrita con mis palabras',     True),
]

for desc, status in checks:
    icon = '✅' if status else '❌'
    print(f'  {icon}  {desc}')

completados = sum(s for _, s in checks)
print(f'\n{completados}/{len(checks)} items completados')

if completados == len(checks):
    print('\n🎉 ¡Listo para hacer el PR en GitHub!')
    print('   Comando: git add . && git commit -m "[S01] Lab Semana 1 - Tu Nombre"')
else:
    print('\n⚠️  Completa todos los items antes de hacer el PR.')